# 01 - Setup and supervised fine-tuning

Accelerator: GPU T4 x2, Internet on. Trains Qwen/Qwen2.5-0.5B on 30,000 UltraChat examples.
Outputs: `outputs/sft/{best,final,metrics.jsonl,config.json,data_summary.json}`.

In [ ]:
import os
import subprocess

from kaggle_secrets import UserSecretsClient

REPO = "small-model-alignment"
GITHUB_USER = "YOUR_GITHUB_USERNAME"
os.chdir("/kaggle/working")
if not os.path.exists(REPO):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    result = subprocess.run(["git", "clone", f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"], capture_output=True, text=True)
    print("cloned" if result.returncode == 0 else result.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://github.com/{GITHUB_USER}/{REPO}.git"])
os.chdir(f"/kaggle/working/{REPO}")
os.environ["PYTHONPATH"] = "."
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!git log --oneline -3

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("visible GPUs:", torch.cuda.device_count())

In [ ]:
!python -m pytest -q tests/

Pilot run: a few optimizer steps on a small subset to check memory, loss, and saving.

In [ ]:
!python scripts/train_sft.py --output_dir outputs/sft_pilot --train_size 256 --val_size 64 --eval_every 4
!rm -rf outputs/sft_pilot

Full run with the guide configuration.

In [ ]:
!python scripts/train_sft.py --output_dir outputs/sft

In [ ]:
!python scripts/plot_metrics.py --out_dir outputs/plots
!cd /kaggle/working && zip -r -q sft_run_outputs.zip small-model-alignment/outputs -x "*.safetensors"
!ls -lh outputs/sft outputs/sft/best